In [1]:
import torch
import torch.nn as nn
from torchvision import models
import loralib as lora
import pandas as pd
from torchvision.models import EfficientNet_V2_S_Weights
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import time
import csv
from torchvision.datasets import ImageFolder
from pathlib import Path
from torchvision.models import efficientnet_v2_s


# Pretrained ResNet50 weights
weights = EfficientNet_V2_S_Weights.IMAGENET1K_V1

# Use ResNet preprocessing
transform = weights.transforms()

VARIANT_ROOT = Path(r"C:\Users\preet\Documents\UBC_MDS\DATA586\Project")

variants = ["blur_little", "blur_medium", "clean", "downsampled", "masked", "noise_rotation"]
# variants = ["clean","blur_medium"]
# get correct mapping
train_dataset = datasets.Food101(root="./data", split="train", download=True)
class_to_idx = train_dataset.class_to_idx

def get_loader(variant):
    dataset = ImageFolder(
        root=VARIANT_ROOT / variant,
        transform=transform
    )

    loader = DataLoader(
        dataset,
        batch_size=64,
        shuffle=False,
        num_workers=8,
        pin_memory=True
    )

    return loader



In [2]:
def build_linear_probe(weights, num_classes=101):
    
    # Load pretrained model
    model = models.efficientnet_v2_s(weights=weights)
    # Freeze everything
    for param in model.parameters():
        param.requires_grad = False
    
    # Replace classifier
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, 101)
    

    return model


# -------------------------
# Small adapter module
# -------------------------
class TSA(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden_dim = max(1, channels // reduction)

        self.adapter = nn.Sequential(
            nn.Conv2d(channels, hidden_dim, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_dim, channels, kernel_size=1)
        )

    def forward(self, x):
        return x + self.adapter(x)  # residual connection


# -------------------------
# EfficientNetV2-S + TSA
# -------------------------
class EfficientNetV2S_TSA(nn.Module):
    def __init__(self, num_classes=101, reduction=16, weights=None):
        super().__init__()

        base = efficientnet_v2_s(weights=weights)

        # Stem
        self.features = self._add_adapters(base.features, reduction)

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(base.classifier[1].in_features, num_classes)

    def _add_adapters(self, features, reduction):
        new_layers = []

        for layer in features:
            if isinstance(layer, nn.Sequential):
                blocks = []
                for block in layer:
                    # Try to infer output channels
                    if hasattr(block, "out_channels"):
                        channels = block.out_channels
                    elif hasattr(block, "conv"):
                        channels = block.conv.out_channels
                    else:
                        # fallback (works for most torchvision blocks)
                        channels = block[-1].out_channels if isinstance(block, nn.Sequential) else None

                    if channels is None:
                        blocks.append(block)
                    else:
                        adapter = TSA(channels, reduction)
                        blocks.append(nn.Sequential(block, adapter))

                new_layers.append(nn.Sequential(*blocks))
            else:
                new_layers.append(layer)

        return nn.Sequential(*new_layers)

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

def build_adapter(weights):
    model = EfficientNetV2S_TSA(num_classes=101).to(device)

    return model

def build_batchnorm(weights, num_classes=101):
    model = models.efficientnet_v2_s(weights=weights)
    
    # Replace classifier (same as before)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, 101)

    return model


    
def apply_lora_outer_layers(model, rank=8, alpha=16, last_n_blocks=2):
    def to_int_if_square(x):
        if isinstance(x, tuple):
            assert x[0] == x[1], f"Non-square kernel not supported: {x}"
            return x[0]
        return x
    def replace_conv(module, parent, name):
        new_layer = lora.Conv2d(
            in_channels=module.in_channels,
            out_channels=module.out_channels,
            kernel_size=to_int_if_square(module.kernel_size),
            stride=to_int_if_square(module.stride),
            padding=to_int_if_square(module.padding),
            dilation=to_int_if_square(module.dilation),
            groups=module.groups,
            bias=(module.bias is not None),
            r=rank,
            lora_alpha=alpha,
        )

        # copy weights
        new_layer.conv.weight.data.copy_(module.weight.data)
        if module.bias is not None:
            new_layer.conv.bias.data.copy_(module.bias.data)

        setattr(parent, name, new_layer)

    def apply_to_block(block):
        for name, module in list(block.named_children()):
            if isinstance(module, nn.Conv2d) and module.groups == 1:
                replace_conv(module, block, name)
            else:
                apply_to_block(module)

    # --- 1. Apply to stem ---
    apply_to_block(model.features[0])

    # --- 2. Apply to last N blocks ---
    for block in model.features[-last_n_blocks:]:
        apply_to_block(block)

def build_efficientnet_bitfit(weights):
    model = models.efficientnet_v2_s(weights=weights)

    # Replace classifier
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, 101)


    return model


In [4]:



    
import os
from pathlib import Path
# BASE_DIR = "Documents/UBC_MDS/DATA586/Project_final/training_scripts/ResNet18/models"    
BASE_DIR = Path.home() / "Documents/UBC_MDS/DATA586/Project_final/training_scripts/EfficientNetV2-S/models"

experiments = [
    {
        "name": "linear_probe",
        "builder": build_linear_probe,
        "ckpt": BASE_DIR / "linear_probe_EffecientNet_best_.pth"
    },
    {
        "name": "batchnorm",
        "builder": build_batchnorm,
        "ckpt": BASE_DIR / "batchnorm_finetune_EfficientNet_best.pth"
    },
    {
        "name": "adapter",
        "builder": build_adapter,
        "ckpt":  BASE_DIR / "efficientnet_tsa_finetune_best.pth"
    },
    {
        "name": "lora",
        "builder": None,
        "ckpt": BASE_DIR / "lora_full_checkpoint_efficient.pth"
    },
        {
        "name": "bitfit",
        "builder": build_efficientnet_bitfit,
        "ckpt": BASE_DIR / "EfficientNetV2S_BitFit_best.pth"
    },
]






device = "cuda" if torch.cuda.is_available() else "cpu"

def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0

    start_time = time.perf_counter()

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            # IMPORTANT for accurate GPU timing
            if device == "cuda":
                torch.cuda.synchronize()

            outputs = model(x)

            if device == "cuda":
                torch.cuda.synchronize()

            preds = outputs.argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    total_time = time.perf_counter() - start_time

    accuracy = correct / total
    time_per_sample = total_time / total
    time_per_batch = total_time / len(loader)

    return accuracy, total_time, time_per_sample, time_per_batch

results = []
for variant in variants:
    print(f"\n=== Variant: {variant} ===")

    test_loader = get_loader(variant)

    for exp in experiments:
        print(f"Running: {exp['name']}")
    
        
    
        ckpt = torch.load(exp["ckpt"], map_location=device)
        
        # handle both cases (robust)
    
        if exp["name"] == "lora":
                # Load pretrained model
                model = models.efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT)
                
                # Replace classifier
                in_features = model.classifier[1].in_features
                model.classifier[1] = nn.Linear(in_features, 101)
                
                # Apply LoRA
                apply_lora_outer_layers(model, rank=8, alpha=16, last_n_blocks=3)
                # print(model.fc.weight.mean(), model.fc.weight.std())
                
                model.load_state_dict(ckpt["model_state_dict"])
                
                model = model.to(device)
                model.eval()
                missing, unexpected = model.load_state_dict(ckpt["model_state_dict"], strict=False)
                # print("Missing:", missing)
                # print("Unexpected:", unexpected)
        
                # print(ckpt.keys())
                # print(model.fc.weight.mean(), model.fc.weight.std())
            
            
    
        else:
            model = exp["builder"](weights).to(device)
            
            # Normal full model loading
            ckpt = torch.load(exp["ckpt"], map_location=device)
            state_dict = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
            model.load_state_dict(state_dict)

        acc, total_time, t_per_sample, t_per_batch = evaluate(model, test_loader, device)
        print(acc)

        results.append({
            "variant": variant,
            "model": exp["name"],
            "accuracy": acc,
            "total_inference_time_sec": total_time,
            "time_per_sample_sec": t_per_sample,
            "time_per_batch_sec": t_per_batch,
        })

df = pd.DataFrame(results)
df.to_csv("results_variants_effcientnetv2s.csv", index=False)

print(df)





=== Variant: blur_little ===
Running: linear_probe


C:\Users\preet\AppData\Local\Temp\ipykernel_30828\4119945860.py:83: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(exp["ckpt"], map_location=device)
C:\User

0.47798019801980196
Running: batchnorm
0.7897029702970297
Running: adapter
0.7610693069306931
Running: lora
0.8038415841584159
Running: bitfit
0.7086336633663366

=== Variant: blur_medium ===
Running: linear_probe
0.28205940594059403
Running: batchnorm
0.6096633663366336
Running: adapter
0.5491089108910892
Running: lora
0.6395643564356436
Running: bitfit
0.5091881188118812

=== Variant: clean ===
Running: linear_probe
0.6226138613861386
Running: batchnorm
0.8632475247524752
Running: adapter
0.8438019801980198
Running: lora
0.8678415841584158
Running: bitfit
0.7923168316831684

=== Variant: downsampled ===
Running: linear_probe
0.09655445544554456
Running: batchnorm
0.29956435643564355
Running: adapter
0.2792871287128713
Running: lora
0.33671287128712873
Running: bitfit
0.239009900990099

=== Variant: masked ===
Running: linear_probe
0.6131881188118812
Running: batchnorm
0.8567920792079208
Running: adapter
0.8372277227722772
Running: lora
0.8623762376237624
Running: bitfit
0.78578217821